# RQ1 — Rail LU TWT: Feature Build & K Sweep

Reads **NBT24TWT_outputs.xlsx**, filters LU-mode stations, builds the
joint-proportional 56-feature matrix, runs KMeans and GMM sweeps for K = 2–15,
and saves diagnostics plus trial cluster labels to `outputs/rq1_trial_numbat_lu_only/`.

**Before running:** if this notebook lives *outside* the FYP project folder,
set `PROJECT_ROOT_RELATIVE` in the Config cell below (e.g. `Path("../FYP")`).
Run `Path.cwd()` in any cell to confirm the current kernel directory.

In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────────────
from pathlib import Path

# Container (Podman) path to the FYP project root — the folder that holds
# PROJECT_CONTEXT.md and 地铁进出站数据/.  This is the in-container path, NOT a
# Windows drive letter (F: / D: do not exist inside the container).
#   - Set to None to auto-discover by walking up from the kernel cwd.
PROJECT_ROOT_RELATIVE = Path("/home/jovyan/work/CASA_FYP/FYP")

# Analysis parameters
MODE_FILTER      = {"LU"}          # keep stations whose mode list contains any of these
K_RANGE          = range(2, 16)
MIN_TOTAL_RAIL   = 50
RANDOM_STATE     = 42
TWT_WINDOW       = ("1800-1815", "0045-0100")   # 18:00 – 01:00, 28 bins

# Print cwd so you can verify the kernel location
print("Kernel working directory:", Path.cwd())
print("Configured project root :", PROJECT_ROOT_RELATIVE)

## 1  Imports

In [ ]:
import re
import json
from datetime import datetime

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score,
)

## 2  Path helpers

In [ ]:
def find_project_root(start: Path | None = None) -> Path:
    """Walk upward from start until PROJECT_CONTEXT.md is found."""
    if start is None:
        try:
            start = Path(__file__).resolve()
        except NameError:          # running in Jupyter – use cwd
            start = Path.cwd().resolve()
    if start.is_file():
        start = start.parent
    for candidate in [start, *start.parents]:
        if (candidate / "PROJECT_CONTEXT.md").exists():
            return candidate
    raise FileNotFoundError(
        "PROJECT_CONTEXT.md not found. "
        "Set PROJECT_ROOT_RELATIVE in the Config cell."
    )


def resolve_project_root(rel: Path | None = None) -> Path:
    if rel is not None:
        # absolute path: use directly; relative path: resolve from cwd
        candidate = rel.resolve() if rel.is_absolute() else (Path.cwd() / rel).resolve()
        if not (candidate / "PROJECT_CONTEXT.md").exists():
            raise FileNotFoundError(
                f"PROJECT_ROOT_RELATIVE resolved to {candidate} "
                "but PROJECT_CONTEXT.md was not found there."
            )
        return candidate
    return find_project_root()


def find_one_file(root: Path, filename: str) -> Path:
    """Find a file by exact name under root, skipping outputs and cache."""
    ignored = {"outputs", "__pycache__", "__MACOSX"}
    matches = [
        p for p in root.rglob(filename)
        if p.is_file()
        and not p.name.startswith("._")
        and not any(part in ignored for part in p.parts)
    ]
    if not matches:
        raise FileNotFoundError(f"{filename} not found under {root}")
    return sorted(matches, key=lambda p: (len(str(p)), str(p)))[0]

## 3  Resolve paths

In [ ]:
ROOT        = resolve_project_root(PROJECT_ROOT_RELATIVE)
NUMBAT_PATH = find_one_file(ROOT, "NBT24TWT_outputs.xlsx")
OUTPUT_DIR  = ROOT / "outputs" / "rq1_trial_numbat_lu_only"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root :", ROOT)
print("NUMBAT file  :", NUMBAT_PATH)
print("Output dir   :", OUTPUT_DIR)

## 4  NUMBAT helper functions

In [ ]:
def detect_time_columns(columns) -> list[str]:
    return [c for c in columns if re.fullmatch(r"\d{4}-\d{4}", str(c))]


def slice_numbat_window(time_cols: list[str], start: str, end: str) -> list[str]:
    """Return the ordered bin labels between start and end, handling midnight wrap."""
    si, ei = time_cols.index(start), time_cols.index(end)
    return time_cols[si:ei + 1] if si <= ei else time_cols[si:] + time_cols[:ei + 1]

## 5  Load station mode lookup

In [ ]:
def load_station_modes(path: Path) -> pd.DataFrame:
    boarders = pd.read_excel(path, sheet_name="Station_Boarders", header=2,
                             usecols=["NLC", "Mode"])
    boarders["NLC"]  = boarders["NLC"].astype(str).str.strip()
    boarders["Mode"] = boarders["Mode"].astype(str).str.strip()

    mode_by_nlc = (
        boarders.dropna(subset=["NLC", "Mode"])
        .drop_duplicates(["NLC", "Mode"])
        .groupby("NLC")
        .agg(modes=("Mode", lambda x: sorted(set(x))))
        .reset_index()
    )
    mode_by_nlc["is_tram"]    = mode_by_nlc["modes"].apply(lambda m: set(m) == {"TRM"})
    mode_by_nlc["has_lu"]     = mode_by_nlc["modes"].apply(lambda m: "LU" in set(m))
    mode_by_nlc["mode_label"] = mode_by_nlc["modes"].apply(lambda m: ",".join(m))
    return mode_by_nlc[["NLC", "mode_label", "is_tram", "has_lu"]]


mode_lookup = load_station_modes(NUMBAT_PATH)

print("Mode filter:", sorted(MODE_FILTER))
print("\nMode label distribution:")
display(mode_lookup["mode_label"].value_counts(dropna=False).sort_index().to_frame())

## 6  Load entry / exit sheets

In [ ]:
def load_station_sheet(path: Path, sheet_name: str, direction: str,
                       mode_lookup: pd.DataFrame) -> pd.DataFrame:
    df = pd.read_excel(path, sheet_name=sheet_name, header=2)
    df.columns = [str(c).strip() for c in df.columns]
    for col in ["NLC", "ASC", "Station", "Fare Zone"]:
        df[col] = df[col].astype(str).str.strip()

    df = df.merge(mode_lookup, on="NLC", how="left", validate="many_to_one")
    df = df[df["has_lu"] == True].copy()

    time_cols  = detect_time_columns(df.columns)
    night_cols = slice_numbat_window(time_cols, *TWT_WINDOW)

    id_cols = ["NLC", "ASC", "Station", "Fare Zone", "mode_label"]
    long_df = df.melt(id_vars=id_cols, value_vars=night_cols,
                      var_name="bin_label", value_name="count")
    long_df["direction"] = direction
    long_df["day_type"]  = "TWT"
    long_df["count"]     = pd.to_numeric(long_df["count"], errors="coerce").fillna(0.0)
    return long_df


entries   = load_station_sheet(NUMBAT_PATH, "Station_Entries", "entry", mode_lookup)
exits     = load_station_sheet(NUMBAT_PATH, "Station_Exits",   "exit",  mode_lookup)
rail_long = pd.concat([entries, exits], ignore_index=True)

print("Entries shape :", entries.shape)
print("Exits shape   :", exits.shape)
print("Combined long :", rail_long.shape)
display(rail_long.head())

## 7  Build feature matrix

In [ ]:
def build_feature_matrix(long_df: pd.DataFrame):
    totals = long_df.groupby(["NLC", "direction"], as_index=False).agg(
        direction_total=("count", "sum")
    )
    df = long_df.merge(totals, on=["NLC", "direction"], how="left")
    df["share"] = np.where(df["direction_total"] > 0,
                           df["count"] / df["direction_total"], 0.0)
    df["feature"] = df["day_type"] + "_" + df["direction"] + "_" + df["bin_label"]

    X = df.pivot_table(index="NLC", columns="feature",
                       values="share", aggfunc="sum", fill_value=0.0)

    meta = (
        long_df.groupby(["NLC", "ASC", "Station", "Fare Zone", "mode_label"],
                        as_index=False)
        .agg(total_rail_activity=("count", "sum"))
    )
    keep_nlc = meta.loc[meta["total_rail_activity"] >= MIN_TOTAL_RAIL, "NLC"]
    dropped  = meta[meta["total_rail_activity"] < MIN_TOTAL_RAIL].copy()

    X    = X.loc[X.index.isin(keep_nlc)].copy()
    meta = meta[meta["NLC"].isin(keep_nlc)].copy()

    row_sums = X.sum(axis=1)
    X = X.div(row_sums.replace(0, np.nan), axis=0).fillna(0.0)
    return X, meta, dropped


X, meta, dropped = build_feature_matrix(rail_long)

print(f"Feature matrix : {X.shape}")
print(f"Stations kept  : {len(meta)}")
print(f"Dropped (low volume): {len(dropped)}")
print(f"Row-sum check  min={X.sum(axis=1).min():.15f}  max={X.sum(axis=1).max():.15f}")
display(meta.head())

## 8  Save intermediate outputs

In [ ]:
X.to_parquet(OUTPUT_DIR / "X_rail_LU_TWT_trial.parquet")
meta.to_csv(OUTPUT_DIR / "rail_LU_TWT_meta_trial.csv", index=False)
dropped.to_csv(OUTPUT_DIR / "rail_LU_TWT_low_volume_dropped.csv", index=False)
print("Saved parquet + meta CSVs to", OUTPUT_DIR)

## 9  K sweep — KMeans

In [ ]:
def cluster_sizes(labels) -> dict:
    values, counts = np.unique(labels, return_counts=True)
    return {str(int(v)): int(c) for v, c in zip(values, counts)}


def run_kmeans_sweep(X: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for k in K_RANGE:
        model  = KMeans(n_clusters=k, n_init=50, random_state=RANDOM_STATE)
        labels = model.fit_predict(X)
        rows.append({
            "model": "kmeans", "k": k,
            "inertia":           model.inertia_,
            "silhouette":        silhouette_score(X, labels),
            "calinski_harabasz": calinski_harabasz_score(X, labels),
            "davies_bouldin":    davies_bouldin_score(X, labels),
            "cluster_sizes":     json.dumps(cluster_sizes(labels)),
        })
    return pd.DataFrame(rows)


print("Running KMeans sweep K=2–15 …")
kmeans_diag = run_kmeans_sweep(X)
kmeans_diag.to_csv(OUTPUT_DIR / "rail_LU_TWT_kmeans_k_diagnostics.csv", index=False)
display(kmeans_diag[["k","silhouette","calinski_harabasz","davies_bouldin","inertia"]])

## 10  K sweep — GMM

In [ ]:
def run_gmm_sweep(X: pd.DataFrame):
    rows, labels_by_k = [], {}
    for k in K_RANGE:
        model = GaussianMixture(
            n_components=k, covariance_type="full",
            n_init=50, tol=1e-6, max_iter=300, random_state=RANDOM_STATE,
        )
        labels    = model.fit_predict(X)
        posterior = model.predict_proba(X)
        max_prob  = posterior.max(axis=1)
        entropy   = -(posterior * np.log(np.clip(posterior, 1e-12, 1.0))).sum(axis=1)
        labels_by_k[k] = labels
        rows.append({
            "model": "gmm", "k": k,
            "bic":                model.bic(X),
            "aic":                model.aic(X),
            "log_likelihood":     model.score(X),
            "converged":          model.converged_,
            "silhouette":         silhouette_score(X, labels),
            "calinski_harabasz":  calinski_harabasz_score(X, labels),
            "davies_bouldin":     davies_bouldin_score(X, labels),
            "mean_max_posterior": float(max_prob.mean()),
            "mean_entropy":       float(entropy.mean()),
            "cluster_sizes":      json.dumps(cluster_sizes(labels)),
        })
    return pd.DataFrame(rows), labels_by_k


print("Running GMM sweep K=2–15 (n_init=50, full covariance) …")
gmm_diag, gmm_labels_by_k = run_gmm_sweep(X)
gmm_diag.to_csv(OUTPUT_DIR / "rail_LU_TWT_gmm_k_diagnostics.csv", index=False)
display(gmm_diag[["k","silhouette","calinski_harabasz","davies_bouldin","bic","aic","converged"]])

## 11  K recommendation

In [ ]:
def recommend_k(diag: pd.DataFrame) -> dict:
    sil_k = diag.loc[diag["silhouette"].idxmax(), "k"]
    ch_k  = diag.loc[diag["calinski_harabasz"].idxmax(), "k"]
    db_k  = diag.loc[diag["davies_bouldin"].idxmin(), "k"]
    return {
        "silhouette_best_k":        int(sil_k),
        "calinski_harabasz_best_k": int(ch_k),
        "davies_bouldin_best_k":    int(db_k),
        "consensus_k":              int(sil_k) if sil_k == ch_k == db_k else None,
    }


recommendation = {"gmm": recommend_k(gmm_diag), "kmeans": recommend_k(kmeans_diag)}

with open(OUTPUT_DIR / "rail_LU_TWT_k_recommendation.json", "w", encoding="utf-8") as f:
    json.dump(recommendation, f, indent=2)

print("GMM  :", recommendation["gmm"])
print("KMeans:", recommendation["kmeans"])

## 12  Export trial labels

In [ ]:
selected_k = recommendation["gmm"]["consensus_k"]
if selected_k is None:
    print("GMM metrics disagree — candidates:", recommendation["gmm"])
    selected_k = recommendation["gmm"]["silhouette_best_k"]
    print(f"Trial export only: using silhouette-best K={selected_k}")

labels    = gmm_labels_by_k[selected_k]
label_df  = meta.copy()
label_df["cluster"] = labels
label_df.to_csv(OUTPUT_DIR / f"rail_LU_TWT_gmm_k{selected_k}_labels_trial.csv", index=False)

sidecar = {
    "run_timestamp":  datetime.now().isoformat(timespec="seconds"),
    "mode":           "rail",
    "mode_filter":    sorted(MODE_FILTER),
    "day_type":       "TWT",
    "window":         TWT_WINDOW,
    "normalisation":  "direction_share_then_joint_row_sum",
    "min_total_rail": MIN_TOTAL_RAIL,
    "k_range":        [min(K_RANGE), max(K_RANGE)],
    "random_state":   RANDOM_STATE,
    "input_file":     str(NUMBAT_PATH),
    "notes": [
        "Prototype uses TWT NUMBAT only.",
        "LU-only trial keeps NLCs whose Station_Boarders mode list contains LU.",
        "No FRI/SAT/SUN day-type comparison available.",
        "No semantic cluster naming is performed.",
    ],
}
with open(OUTPUT_DIR / "rail_LU_TWT_trial_sidecar.json", "w", encoding="utf-8") as f:
    json.dump(sidecar, f, indent=2)

print(f"Saved K={selected_k} trial labels + sidecar to:", OUTPUT_DIR)
display(label_df.head())